# Milestone 2 — Transformers, Embeddings & Zero-Shot Ranking
**Smart MCQ Solver Challenge** · Introduction to DL and GenAI Project

**Name:** Vihaan Bhambhani\
**Roll No:** 24f1002825\
**Term:** T2-2026



In [1]:
!pip install -q transformers datasets sentence-transformers
import os, torch
import numpy as np
from datasets import load_dataset

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
for alt in ["/kaggle/input/smart-mcq-solver-challenge", "."]:
    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        DATA_DIR = alt
TRAIN = f"{DATA_DIR}/train.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| data:", TRAIN)

device: cuda | data: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv


## Q1 - HF `datasets` + `.map()` -> `combined_text` length at index 51
Load with `datasets` (not pandas), build `combined_text = prompt + " " + A`, take `len()` at zero-indexed row 51.

In [2]:
ds = load_dataset("csv", data_files=TRAIN)["train"]
ds = ds.map(lambda x: {"combined_text": f"{x['prompt']} {x['A']}"})
q1 = len(ds[51]["combined_text"])
print("Q1 ANSWER:", q1)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 ANSWER: 614


## Q2 - `bert-base-uncased` tokenizer vocabulary size

In [3]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
q2 = tok.vocab_size
print("Q2 ANSWER:", q2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 ANSWER: 30522


## Q3 - `[SEP]` token integer ID

In [4]:
q3 = tok.sep_token_id
print("Q3 ANSWER:", q3, "| sanity:", tok.convert_tokens_to_ids("[SEP]"))

Q3 ANSWER: 102 | sanity: 102


## Q4 - `input_ids` tensor shape (all prompts, max_length=128)

In [5]:
enc_all = tok(list(ds["prompt"]), padding="max_length", truncation=True,
              max_length=128, return_tensors="pt")
q4 = tuple(enc_all["input_ids"].shape)
print("Q4 ANSWER:", q4)

Q4 ANSWER: (2000, 128)


## Q5 - Per-attention-head dimensionality
Hidden size 768 split equally across 12 heads.

In [6]:
q5 = 768 // 12
print("Q5 ANSWER:", q5) 

Q5 ANSWER: 64


## Q6 - `last_hidden_state` shape for row index 0 prompt (default tokenization)
`id` starts at 1, so "row ID 0" = index 0 (first row).

In [7]:
from transformers import AutoModel
bert = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()
enc0 = tok(ds[0]["prompt"], return_tensors="pt").to(device)
with torch.no_grad():
    out0 = bert(**enc0)
q6 = tuple(out0.last_hidden_state.shape)
print("Q6 ANSWER:", q6)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6 ANSWER: (1, 31, 768)


## Q7 - Sum of first 5 values of the `[CLS]` embedding (4 dp)

In [8]:
cls_vec = out0.last_hidden_state[0, 0]        
q7 = round(cls_vec[:5].sum().item(), 4)
print("Q7 ANSWER:", q7)

Q7 ANSWER: -1.2001


## Q8 - Attention weight `[CLS]` -> `fusion`, last layer, head 0 (4 dp)
String: `"Light-ion fusion is a technique."`

In [9]:
bert_attn = AutoModel.from_pretrained("bert-base-uncased",
                                      output_attentions=True).to(device).eval()
enc = tok("Light-ion fusion is a technique.", return_tensors="pt").to(device)
with torch.no_grad():
    out = bert_attn(**enc)
toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
fusion_idx = toks.index("fusion") if "fusion" in toks else \
             next(i for i,t in enumerate(toks) if "fusion" in t.replace("##",""))
attn_last_head0 = out.attentions[-1][0, 0]        # (seq, seq)
q8 = round(attn_last_head0[0, fusion_idx].item(), 4)   # row 0 = CLS query
print("tokens:", toks)
print("fusion at index:", fusion_idx)
print("Q8 ANSWER:", q8)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion at index: 4
Q8 ANSWER: 0.1025


## Q9 - MiniLM cosine similarity: prompt vs Option B, row index 0 (4 dp)

In [10]:
from sentence_transformers import SentenceTransformer, util
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
e_p = st.encode(ds[0]["prompt"], convert_to_tensor=True)
e_b = st.encode(ds[0]["B"],     convert_to_tensor=True)
q9 = round(util.cos_sim(e_p, e_b).item(), 4)
print("Q9 ANSWER:", q9)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 ANSWER: 0.7658


## Q10 - MiniLM ranking pipeline: MAP@3, and TF-IDF-miss / MiniLM-hit count
Pipeline 1 = TF-IDF cosine (Milestone 1). Pipeline 2 = MiniLM embeddings cosine.

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
LABELS = ["A","B","C","D","E"]
df = pd.read_csv(TRAIN)

def map_at_3(truth, preds):
    for i,p in enumerate(preds[:3]):
        if p == truth: return 1.0/(i+1)
    return 0.0

prompt_emb = st.encode(df["prompt"].tolist(), convert_to_tensor=True,
                       batch_size=128, show_progress_bar=True)
opt_emb = {c: st.encode(df[c].astype(str).tolist(), convert_to_tensor=True,
                        batch_size=128) for c in LABELS}
mini_sims = np.column_stack([
    util.cos_sim(prompt_emb, opt_emb[c]).diagonal().cpu().numpy() for c in LABELS])
mini_top3 = [[LABELS[k] for k in np.argsort(-row)[:3]] for row in mini_sims]
q10_map3 = np.mean([map_at_3(a,t) for a,t in zip(df["answer"], mini_top3)])

corpus = list(df["prompt"].astype(str))
for c in LABELS: corpus += list(df[c].astype(str))
vec = TfidfVectorizer(stop_words="english").fit(corpus)
P = vec.transform(df["prompt"].astype(str))
tfidf_sims = np.column_stack([
    cosine_similarity(P, vec.transform(df[c].astype(str))).diagonal() for c in LABELS])
tfidf_top3 = [[LABELS[k] for k in np.argsort(-row)[:3]] for row in tfidf_sims]

count = sum((df["answer"].iloc[i] not in tfidf_top3[i]) and
            (df["answer"].iloc[i] in mini_top3[i]) for i in range(len(df)))
print(f"Q10 ANSWER part 1 (MiniLM MAP@3): {q10_map3:.4f}")
print(f"Q10 ANSWER part 2 (TF-IDF miss & MiniLM hit count): {count}")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Q10 ANSWER part 1 (MiniLM MAP@3): 0.4231
Q10 ANSWER part 2 (TF-IDF miss & MiniLM hit count): 493


## Q11 - Zero-shot classification (bart-large-mnli), row index 1, top prob (4 dp)
Candidate labels = Options A, B, C.

In [12]:
from transformers import pipeline
zs = pipeline("zero-shot-classification",
              device=0 if device=="cuda" else -1)
row1 = df.iloc[1]
labels = [str(row1["A"]), str(row1["B"]), str(row1["C"])]
r_soft = zs(str(row1["prompt"]), candidate_labels=labels)
q11 = round(r_soft["scores"][0], 4)
print("Q11 ANSWER:", q11)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11 ANSWER: 0.4575


## Q12 - Same with `multi_label=True`: |sum(softmax) - sum(sigmoid)| (4 dp)

In [13]:
r_multi = zs(str(row1["prompt"]), candidate_labels=labels, multi_label=True)
q12 = round(abs(sum(r_soft["scores"]) - sum(r_multi["scores"])), 4)
print("sum softmax:", round(sum(r_soft['scores']),4), "| sum sigmoid:", round(sum(r_multi['scores']),4))
print("Q12 ANSWER:", q12)

sum softmax: 1.0 | sum sigmoid: 0.0005
Q12 ANSWER: 0.9995


## Q13 - Generative QA with flan-t5-small, row index 0
Exact prompt template, `max_new_tokens=5`.

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(device).eval()

row0 = df.iloc[0]
s = (f"Question: {row0['prompt']}. Is the correct answer "
     f"A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B.")

inp = t5_tok(s, return_tensors="pt").to(device)
with torch.no_grad():
    gen_ids = t5.generate(**inp, max_new_tokens=5)
q13 = t5_tok.decode(gen_ids[0], skip_special_tokens=True)
print("Q13 ANSWER (exact string):", repr(q13))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 ANSWER (exact string): 'B'


## Answer summary

In [15]:
print("Milestone 2 - answers")
print(f"Q1  combined_text len @ idx 51 : {q1}")
print(f"Q2  bert vocab size            : {q2}")
print(f"Q3  [SEP] id                   : {q3}")
print(f"Q4  input_ids shape            : {q4}")
print(f"Q5  per-head dim               : {q5}")
print(f"Q6  last_hidden_state shape    : {q6}")
print(f"Q7  sum first5 CLS             : {q7}")
print(f"Q8  attn CLS->fusion           : {q8}")
print(f"Q9  MiniLM cos(prompt,B) r0    : {q9}")
print(f"Q10 MiniLM MAP@3 / miss-hit cnt: {q10_map3:.4f} / {count}")
print(f"Q11 zero-shot top prob         : {q11}")
print(f"Q12 |softmax-sigmoid| sum diff : {q12}")
print(f"Q13 flan-t5 output             : {q13!r}")

Milestone 2 - answers
Q1  combined_text len @ idx 51 : 614
Q2  bert vocab size            : 30522
Q3  [SEP] id                   : 102
Q4  input_ids shape            : (2000, 128)
Q5  per-head dim               : 64
Q6  last_hidden_state shape    : (1, 31, 768)
Q7  sum first5 CLS             : -1.2001
Q8  attn CLS->fusion           : 0.1025
Q9  MiniLM cos(prompt,B) r0    : 0.7658
Q10 MiniLM MAP@3 / miss-hit cnt: 0.4231 / 493
Q11 zero-shot top prob         : 0.4575
Q12 |softmax-sigmoid| sum diff : 0.9995
Q13 flan-t5 output             : 'B'
